# 00 · Cuaderno maestro / *Master notebook*
**GUAIRA-SAR** · SRL *2026 Venezuela, Colombia and Panama Earthquakes*

🇪🇸 **Un solo cuaderno** que llama a todo el flujo (inventario → descargas → InSAR → daño → figuras) usando el código de GitHub (`src/`). Cada paso se ejecuta **una vez**: queda registrado en Drive (`05_documentacion/pasos_ejecutados.csv`) y se salta solo la próxima vez. Los pasos ya hechos el 19-nov figuran como ✅ *previo*. Para repetir uno a propósito: `P("id", forzar=True)`.
🇬🇧 **One notebook** that calls the whole workflow using the GitHub code. Each step runs **once**; it is logged on Drive and skipped next time. Steps done on 19 Nov show as ✅ *previo*. To repeat on purpose: `P("id", forzar=True)`.

⚠️ 🇪🇸 **HyP3:** el envío identifica trabajos por *par de escenas + resolución* (no por nombre) y **solo envía lo que no existe**. Hoy solo falta 033A a 40 m.
⚠️ 🇬🇧 **HyP3:** submissions are matched by *scene pair + resolution*; only what does not exist is sent. Today only 033A at 40 m is missing.
🔒 🇪🇸 Credenciales en **Secretos de Colab** (🔑 en la barra lateral): `EARTHDATA_USER` y `EARTHDATA_PASS`. No se escriben en el cuaderno. 🇬🇧 Credentials in **Colab Secrets**; never typed into the notebook.
📝 Contexto completo / full context: `docs/HANDOFF.md`.

## 0 · Preparar entorno / *Setup*

In [ ]:
NOTEBOOK = "00_Maestro"
!pip install -q asf_search hyp3_sdk rasterio geopandas shapely contextily pyarrow duckdb boto3 pystac-client requests h5py pyproj scipy huggingface_hub cmcrameri
import os, subprocess, datetime
REPO = "/content/SAR_multifrequency"
CUADERNO_VERSION = "2026-11-27.2"      # versión de ESTE cuaderno / version of THIS notebook
def _git(*a):
    return subprocess.run(["git", "-C", REPO, *a], capture_output=True, text=True).stdout.strip()
if os.path.isdir(REPO):
    _antes = _git("rev-parse", "HEAD")
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
    _log = _git("log", "--oneline", "--no-merges", f"{_antes}..HEAD") if _antes else ""
    print("Cambios nuevos en GitHub desde la última vez / new GitHub changes since last time:")
    print(_log[:1800] if _log else "   (ninguno / none)")
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/jf-floresriera/SAR_multifrequency", REPO], check=True)
_vf = f"{REPO}/notebooks/VERSION_CUADERNO.txt"
_v = open(_vf).read().strip() if os.path.exists(_vf) else ""
if _v and _v != CUADERNO_VERSION:
    print()
    print(f"⚠️ ESTE cuaderno es la versión {CUADERNO_VERSION} y en GitHub hay la {_v}: puede faltar alguna celda nueva.")
    print("   Ábralo de nuevo desde / reopen from: https://colab.research.google.com/github/jf-floresriera/SAR_multifrequency/blob/main/notebooks/00_Maestro.ipynb")
    print("   Mientras tanto, cualquier paso se corre en una celda nueva con: P(\"id_del_paso\", forzar=True)  (ids al final / ids at the end)")
SRC = f"{REPO}/src"
VERSION = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()

from google.colab import drive
drive.mount("/content/drive")
RAIZ = "/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR"
TRABAJO = "/content/drive/My Drive/InSAR_LaGuaira_Resultados/inventario_2026"
for d in (TRABAJO, f"{RAIZ}/05_documentacion"):
    os.makedirs(d, exist_ok=True)
os.chdir(TRABAJO)
with open(f"{RAIZ}/05_documentacion/versiones_codigo.csv", "a") as f:   # trazabilidad / traceability
    f.write(f"{datetime.datetime.now():%Y-%m-%d %H:%M},{NOTEBOOK},{VERSION}\n")
import sys, importlib
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import comun, maestro; importlib.reload(comun); importlib.reload(maestro)
LOGS = f"{RAIZ}/05_documentacion/logs"
def correr(script):
    """Ejecuta src/<script> y guarda todo lo impreso en 05_documentacion/logs / runs and logs everything."""
    return comun.correr(script, LOGS, SRC, NOTEBOOK, VERSION)
def P(pid, **kw):
    """Ejecuta un paso del flujo salvo que ya esté hecho / run a workflow step unless already done."""
    return maestro.paso(pid, correr, RAIZ, VERSION, **kw)
print(f"Código / code: {VERSION} · Carpeta / folder: {os.getcwd()}")
print("Pasos disponibles / available steps:", maestro.IDS)

## 1 · Credenciales NASA Earthdata (Secretos de Colab) / *Credentials (Colab Secrets)*

In [ ]:
import asf_search as asf
try:
    from google.colab import userdata
    usuario, clave = userdata.get("EARTHDATA_USER"), userdata.get("EARTHDATA_PASS")
except Exception:
    import getpass
    print("Sin Secretos de Colab: se pedirán aquí (la clave no se muestra). / No Colab Secrets: prompting here.")
    usuario = input("Usuario Earthdata / username: "); clave = getpass.getpass("Contraseña / password: ")
session = asf.ASFSession().auth_with_creds(usuario, clave)
print("Autenticación correcta / authenticated")

## 2 · Panel de estado / *Status board*
✅ hecho · ♻️ se puede correr siempre · ⏳ pendiente

In [ ]:
import pandas as pd; pd.set_option("display.max_colwidth", 90, "display.width", 200)
maestro.estado(RAIZ)

## A · Inventario multisensor / *Multi-sensor inventory*
🇪🇸 Umbra (X), NISAR y ALOS-2 (L), Sentinel-1 (C), Planet. Estas figuras y tablas van al **material suplementario** (S1–S3); una versión sintética alimenta la Fig. 1. 🇬🇧 These go to the **electronic supplement**.

In [ ]:
P("inv_umbra")
P("inv_nisar")

In [ ]:
P("inv_s1", variables={"ENVIAR_HYP3": False})   # solo busca / search only

In [ ]:
P("inv_alos2")   # ALOS-2: si ASF no lo tiene, ver docs/HANDOFF.md §5 (fecha de corte 5-dic) / see HANDOFF §5

In [ ]:
P("inv_tabla")
P("mapas_base")   # se repite solo si cambia su código / re-runs by itself when its code changes

## B · HyP3 y descargas / *HyP3 and downloads*
🇪🇸 `hyp3_estado` descarga lo terminado a Drive (HyP3 borra a los ~14 días). `hyp3_envio` está **bloqueado** hasta poner `confirmar=True`: enviará SOLO lo que no exista (hoy: 033A a 40 m, 1 trabajo). 🇬🇧 `hyp3_envio` is locked until `confirmar=True`.

In [ ]:
P("hyp3_estado", variables={"DESCARGAR": True})

In [ ]:
P("hyp3_envio", confirmar=False, variables={"ENVIAR_HYP3": True})   # ← cambiar a confirmar=True para enviar / set True to submit

In [ ]:
P("descargas", variables={"ESPERAR_HYP3": False})   # NISAR (ya en Drive se omite), Umbra GEC, Sentinel-1

## C · InSAR banda C y L / *C- and L-band InSAR*
🇪🇸 Escriba las coordenadas de CCS1 (ficha NGL / IGVSB) para validar con GNSS. 🇬🇧 Enter CCS1 coordinates to validate.

In [ ]:
P("auxiliares")
P("nisar_tif")
P("s1_tif")

In [ ]:
P("descomp", variables={"CCS1_LON": None, "CCS1_LAT": None})   # ej./e.g. -66.88, 10.49 (verificar)

### C0 · Auditoría de correcciones / Corrections audit
🇪🇸 LEE cada GUNW de NISAR y cada ZIP de HyP3 y lista qué correcciones trae (ionosfera, troposfera, mareas, DEM…). Ver docs/CORRECCIONES_INSAR.md.
🇬🇧 READS every NISAR GUNW and HyP3 ZIP and lists which corrections it carries. See docs/CORRECCIONES_INSAR.md.

In [ ]:
P("auditoria")

### 🔁 Reproducir TODO (paneles individuales **y** mosaicos) / Reproduce EVERYTHING (individual panels **and** mosaics)
🇪🇸 Orden: `nisar_tif`/`s1_tif` → `descomp` → `auditoria` → `ucsd_los` → **`mosaico`** → `qc` → `comparacion` → `insar_metricas`.
- **Paneles individuales (se conservan):** `qc` (una figura por interferograma: LOS en mm, fase en rad, franjas, coherencia, histogramas) e `insar_metricas` (`insar_comparable_propios_*`, `insar_comparable_ucsd_*`, perfil, tablas de métricas).
- **Mosaicos (adicionales):** `mosaico` (une frames/tracks vecinos de la misma familia y dirección; escribe también la procedencia de cada pieza), el `qc` de cada mosaico (`qc_*_MOSAICO_*`) y `comparacion` (figura principal con todos los modelos).
- Texto principal = mosaicos + comparación; **suplemento = paneles individuales y QC**.
🇬🇧 Individual panels (kept): `qc` and `insar_metricas`. Mosaics (additional): `mosaico`, mosaic QC and `comparacion`. Main text = mosaics + comparison; supplement = individual panels and QC.

### C1b · ALOS-2 y NISAR procesados por UCSD / UCSD-processed ALOS-2 and NISAR
🇪🇸 Descarga 7 archivos de topex.ucsd.edu (~445 MB cada uno, ~3 GB en total; se guardan en 01_datos_crudos/ucsd_los) y los convierte a GeoTIFF sobre nuestra malla. Decide el signo comparando con NUESTRO NISAR. **No redistribuir** (licencia no declarada): citar a UCSD/Scripps. Tarda unos minutos.
🇬🇧 Downloads 7 files from topex.ucsd.edu (~445 MB each, ~3 GB total) and converts them to GeoTIFF on our grid; sign decided by comparison with OUR NISAR. **Do not redistribute** (licence not stated): cite UCSD/Scripps.

In [ ]:
P("ucsd_los")   # prueba rápida / quick test: variables={"SOLO_TRACKS": ["NSR_T162A"]}

### C1c · Mosaico de frames/tracks NISAR vecinos / Mosaic of adjacent NISAR frames/tracks
🇪🇸 Une los frames y tracks de la misma dirección (ASC: T162A+T061A; DESC: T054D+T126D, como hace UCSD), **igualando el desfase en el solape**, y mide la costura (RMS). Es para VER y comparar; Este/Vertical sigue usando los productos separados.
🇬🇧 Merges same-direction frames/tracks (ASC: T162A+T061A; DESC: T054D+T126D, as UCSD does), **matching the offset in the overlap**, and reports the seam (RMS). For viewing/comparison; East/Up still uses the separate products.

In [ ]:
P("mosaico")

### C0b · Control de calidad de cada interferograma / Per-interferogram QC
🇪🇸 ¿La deformación realmente se ve? Muestra LOS con estiramiento robusto, **franjas** (fase re-envuelta), coherencia, histograma, y escribe un diagnóstico por producto. Ejecutar ANTES de interpretar mapas.
🇬🇧 Is the deformation really visible? Robust-stretch LOS, **fringes** (re-wrapped phase), coherence, histogram and a per-product diagnosis. Run BEFORE interpreting maps.

In [ ]:
P("qc")

### C1d · Figura de comparación de todos los modelos / Comparison figure of all models
🇪🇸 Filas = familia de sensor (Sentinel-1, NISAR, NISAR-UCSD, ALOS-2-UCSD), columnas = ASC | DESC; misma ventana y escala. **Figura principal**; los productos individuales y el QC van al suplemento.
🇬🇧 Rows = sensor family, columns = ASC | DESC; same window and scale. **Main figure**; individual products and QC go to the supplement.

In [ ]:
P("comparacion")   # Fig. 3 (franjas) y Fig. 4 (LOS) · regla de selección: variables={"UMBRAL_COSTURA_MM": 40, "LIM_MM": 400}

### C2 · Métricas y gráficos comparables de toda la interferometría / Comparable metrics and plots for all InSAR
🇪🇸 Misma ventana, misma escala de color (± cm común) y mismo contexto para todos los productos (Sentinel-1 y NISAR, incluido el par pre-sísmico de control). Genera la tabla de métricas (CSV ES/EN), mapas regional y costa, y un perfil común.
🇬🇧 Same window, same colour scale (common ± cm) and same context for every product (Sentinel-1 and NISAR, including the pre-seismic control pair). Produces the metrics table (CSV ES/EN), regional and coastal maps, and a common profile.

In [ ]:
P("insar_metricas")   # opcional / optional: variables={"LIM_MM": 300, "COH_MIN": 0.3}

### C3 · Tablas del artículo / Paper tables
🇪🇸 Tabla 1 (datos), Tabla 2 (procesamiento y correcciones), S1 (calidad con **tierra vs mar**), S2 (mosaicos), S3 (C vs L en píxeles comunes sobre tierra; propio vs UCSD). CSV + Markdown, ES y EN, en `05_documentacion/tablas/`.
🇬🇧 Tables 1, 2, S1 (quality, land vs sea), S2 (mosaics), S3 (C vs L on common land pixels; ours vs UCSD). CSV + Markdown, ES and EN.

In [ ]:
P("tablas")

### C4 · Figura de inventario (Fig. 2) / Inventory figure
🇪🇸 Línea de tiempo por banda (Umbra X, Sentinel-1 C, NISAR L, **ALOS-2 L**, óptico) con escenas y pares usados. Lista completa de figuras: `docs/LISTA_FIGURAS.md`.
🇬🇧 Timeline per band incl. ALOS-2. Full figure list: `docs/LISTA_FIGURAS.md`.

In [ ]:
P("inventario")

## D · Daño: Umbra X + Planet + referencia / *Damage*
🇪🇸 Las figuras por sitio del 19-nov **no son publicables** (jet, título interno, Esri): tras actualizar `umbra_planet_sitios.py` al estilo nuevo use `forzar=True`. 🇬🇧 The 19 Nov per-site figures are not publishable; re-run with `forzar=True` after the style update.

### D2 · Óptico de alta resolución antes/después (Vantor abierto) / High-resolution optical before/after (open Vantor)
🇪🇸 Busca escenas de ~0,4 m anteriores y posteriores al sismo sobre cada sitio (sin descargas completas) y guarda un CSV y recortes de prueba. Licencia CC BY-NC 4.0: no subir a GitHub.
🇬🇧 Finds ~0.4 m pre- and post-event scenes over each site (no full downloads) and saves a CSV and test chips. CC BY-NC 4.0: do not commit.

In [ ]:
P("vantor", variables={"SITIOS_VANTOR": {"Caracas_Este_Petare": (10.4835, -66.8070)},   # (lat, lon) ← cambiar por el centro real / replace with true centre
                                     "LADO_M": 1000})

In [ ]:
P("danio_ref")

In [ ]:
P("danio_sitios", variables={"LADO_M": 1000, "ZOOM_M": 400, "SITIOS_A_PROCESAR": ["Catia_La_Mar", "Playa_Grande_Urimare", "La_Guaira"]})

## E · Figuras y tablas finales / *Final figures and tables*
🇪🇸 **Texto principal (≤ 10 figuras, ≤ 3 tablas):** Fig. 1 ubicación y cobertura · Fig. 3–5 LOS y Este/Vertical C vs L · Fig. 6 coherencia por cobertura · Fig. 7 ventana postsísmica · Fig. 8 daño Umbra × Planet · Fig. 9 daño vs gradiente/distancia a la falla · Fig. 10 infraestructura · Tablas 1–3.
**Suplemento (sin límite):** S1 línea de tiempo · S2 huellas por sensor (Sentinel-1, NISAR, ALOS-2, Umbra, Planet) · Tabla S1 inventario completo · S3 los 6 sitios de daño · S4 paletas y pruebas de gris.
🇬🇧 Scripts for Figs. 6, 7, 9, 10 and Table 3 are **pending**; they will be added to `src/` and as steps of this notebook (`PASOS` in `src/maestro.py`).

In [ ]:
P("exportar")   # figuras a GitHub: NO subir imágenes Planet (CC BY-NC) ni Esri / do not upload Planet or Esri imagery

## Estado final / *Final status*

In [ ]:
maestro.estado(RAIZ)